In [ ]:
# =========================
# Section 1: Install and import libraries
# What this section is doing:
# - Imports the libraries needed to upload, read, clean, calculate, and export the Excel data.
# =========================

import pandas as pd
import numpy as np
from google.colab import files

In [ ]:
# =========================
# Section 2: Upload the input Excel file
# What this section is doing:
# - Uploads your file: houston_data_centers_final.xlsx
# - Reads it into a pandas DataFrame
# =========================

uploaded = files.upload()

input_file = "houston_data_centers_final.xlsx"
df = pd.read_excel(input_file)

print("Raw data shape:", df.shape)
df.head()

Saving houston_data_centers_final.xlsx to houston_data_centers_final (1).xlsx
Raw data shape: (120, 5)


,Data_Center_Name,Address,Capacity_MW,Area_SqFt,Source_URL
0,TRG Houston One HOU1TRG Datacenters2626 Spring...,TRG Houston One HOU1,24 MW,"150,000 sq.f",https://www.datacentermap.com/usa/texas/housto...
1,Netrality - 1301 FanninNetrality1301 Fannin St...,TRG Houston One HOU1,26 MW,"231,700 sq.f",https://www.datacentermap.com/usa/texas/housto...
2,TRG Houston Two HOU2TRG Datacenters2626 Spring...,TRG Houston One HOU1,24 MW,110000 sq.f.,https://www.datacentermap.com/usa/texas/housto...
3,TRG Houston CampusTRG Datacenters2626 Spring C...,TRG Houston One HOU1,30 MW,NaN,https://www.datacentermap.com/usa/texas/housto...
4,12235 North Fwy (IAH14)Digital Realty12235 Nor...,TRG Houston One HOU1,6.2 MW,"42,400 sq.f",https://www.datacentermap.com/usa/texas/housto...


In [ ]:
# =========================
# Section 3: Inspect column names
# What this section is doing:
# - Displays the column names so you can confirm the exact headers in your Excel file
# - Update COL_NAME and COL_CAPACITY below if needed
# =========================

print("Columns in the uploaded file:")
print(df.columns.tolist())

Columns in the uploaded file:
['Data_Center_Name', 'Address', 'Capacity_MW', 'Area_SqFt', 'Source_URL']


In [ ]:
# =========================
# Section 4: Define the columns to use
# What this section is doing:
# - Sets the data center name column
# - Sets the Fully Built-Out Power / capacity column
# - Change these only if your actual Excel headers are slightly different
# =========================

COL_NAME = "Data_Center_Name"
COL_CAPACITY = "Capacity_MW"

In [ ]:
# =========================
# Section 5: Clean the data
# What this section is doing:
# - Keeps only name and capacity
# - Removes duplicate data centers based on name
# - Removes rows where capacity is missing or N/A
# - Converts capacity to numeric
# =========================

clean_df = df[[COL_NAME, COL_CAPACITY]].copy()

# Standardize text
clean_df[COL_NAME] = clean_df[COL_NAME].astype(str).str.strip()
clean_df[COL_CAPACITY] = clean_df[COL_CAPACITY].astype(str).str.strip()

raw_rows = len(clean_df)

# Remove obvious missing values
missing_tokens = ["", "N/A", "NA", "na", "n/a", "None", "none", "-", "--"]
clean_df = clean_df[~clean_df[COL_CAPACITY].isin(missing_tokens)].copy()

# Remove duplicates by data center name
before_dupes = len(clean_df)
clean_df = clean_df.drop_duplicates(subset=[COL_NAME], keep="first").copy()
after_dupes = len(clean_df)

# Convert capacity to numeric
clean_df[COL_CAPACITY] = (
    clean_df[COL_CAPACITY]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.extract(r"([0-9]*\.?[0-9]+)")[0]
)

clean_df[COL_CAPACITY] = pd.to_numeric(clean_df[COL_CAPACITY], errors="coerce")

# Drop rows where numeric conversion failed
before_numeric_drop = len(clean_df)
clean_df = clean_df.dropna(subset=[COL_CAPACITY]).copy()

# Remove zero or negative capacity values if any
clean_df = clean_df[clean_df[COL_CAPACITY] > 0].copy()

clean_df = clean_df.rename(columns={
    COL_NAME: "Data Center Name",
    COL_CAPACITY: "Capacity MW"
}).reset_index(drop=True)

print("Raw rows:", raw_rows)
print("After duplicate removal:", after_dupes)
print("After numeric/missing capacity cleanup:", len(clean_df))

clean_df.head()

Raw rows: 120
After duplicate removal: 120
After numeric/missing capacity cleanup: 52


,Data Center Name,Capacity MW
0,TRG Houston One HOU1TRG Datacenters2626 Spring...,24.0
1,Netrality - 1301 FanninNetrality1301 Fannin St...,26.0
2,TRG Houston Two HOU2TRG Datacenters2626 Spring...,24.0
3,TRG Houston CampusTRG Datacenters2626 Spring C...,30.0
4,12235 North Fwy (IAH14)Digital Realty12235 Nor...,6.2


In [ ]:
# =========================
# Section 6: Set model assumptions
# What this section is doing:
# - Defines the constants used in the cooling water estimation model
# - Capacity MW is treated as Fully Built-Out Power from DataCenterMap
# =========================

PUE = 1.58
WUE = 2.2  # liters per kWh

UTIL_LOW = 0.45
UTIL_MID = 0.60
UTIL_HIGH = 0.75

In [ ]:
# =========================
# Section 7: Calculate water consumption
# What this section is doing:
# - Converts total facility power to IT load using PUE
# - Applies low/mid/high utilization
# - Converts annual IT energy into annual water consumption using WUE
# - Converts Liters to Gallons (1 L = 0.264172 Gallons)
# =========================

L_TO_GAL = 0.264172
result_df = clean_df.copy()

# Step 1: Convert capacity MW to IT load MW
result_df["IT Load MW"] = result_df["Capacity MW"] / PUE

# Step 2: Apply utilization scenarios
result_df["IT Load Low MW"] = result_df["IT Load MW"] * UTIL_LOW
result_df["IT Load Mid MW"] = result_df["IT Load MW"] * UTIL_MID
result_df["IT Load High MW"] = result_df["IT Load MW"] * UTIL_HIGH

# Step 3: Convert MW to annual kWh
result_df["Annual kWh Low"] = result_df["IT Load Low MW"] * 1000 * 8760
result_df["Annual kWh Mid"] = result_df["IT Load Mid MW"] * 1000 * 8760
result_df["Annual kWh High"] = result_df["IT Load High MW"] * 1000 * 8760

# Step 4: Water consumption in liters/year
result_df["Water Consumption Low (L/year)"] = result_df["Annual kWh Low"] * WUE
result_df["Water Consumption Mid (L/year)"] = result_df["Annual kWh Mid"] * WUE
result_df["Water Consumption High (L/year)"] = result_df["Annual kWh High"] * WUE

# Step 5: Convert Liters to Gallons
result_df["Water Consumption Low (Gal/year)"] = result_df["Water Consumption Low (L/year)"] * L_TO_GAL
result_df["Water Consumption Mid (Gal/year)"] = result_df["Water Consumption Mid (L/year)"] * L_TO_GAL
result_df["Water Consumption High (Gal/year)"] = result_df["Water Consumption High (L/year)"] * L_TO_GAL

# Keep requested columns
final_df = result_df[[
    "Data Center Name",
    "Capacity MW",
    "Water Consumption Low (L/year)",
    "Water Consumption Mid (L/year)",
    "Water Consumption High (L/year)",
    "Water Consumption Low (Gal/year)",
    "Water Consumption Mid (Gal/year)",
    "Water Consumption High (Gal/year)"
]].copy()

# Formatting
cols_to_round = final_df.columns.drop("Data Center Name")
final_df[cols_to_round] = final_df[cols_to_round].round(2)

display(final_df.head())

,Data Center Name,Capacity MW,Water Consumption Low (L/year),Water Consumption Mid (L/year),Water Consumption High (L/year),Water Consumption Low (Gal/year),Water Consumption Mid (Gal/year),Water Consumption High (Gal/year)
0,TRG Houston One HOU1TRG Datacenters2626 Spring...,24.0,1.317327e+08,1.756435e+08,2.195544e+08,34800079.79,46400106.39,58000132.98
1,Netrality - 1301 FanninNetrality1301 Fannin St...,26.0,1.427104e+08,1.902805e+08,2.378506e+08,37700086.44,50266781.92,62833477.40
2,TRG Houston Two HOU2TRG Datacenters2626 Spring...,24.0,1.317327e+08,1.756435e+08,2.195544e+08,34800079.79,46400106.39,58000132.98
3,TRG Houston CampusTRG Datacenters2626 Spring C...,30.0,1.646658e+08,2.195544e+08,2.744430e+08,43500099.74,58000132.98,72500166.23
4,12235 North Fwy (IAH14)Digital Realty12235 Nor...,6.2,3.403094e+07,4.537458e+07,5.671823e+07,8990020.61,11986694.15,14983367.69


In [ ]:
# =========================
# Section 8: Export the cleaned and calculated data
# What this section is doing:
# - Saves the final results to a new Excel file
# - Downloads the file to your computer
# =========================

output_file = "Houston_datacenter_cooling_estimate_data.xlsx"

final_df.to_excel(output_file, index=False)

print(f"Saved file: {output_file}")
files.download(output_file)

Saved file: Houston_datacenter_cooling_estimate_data.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# =========================
# Section 9: Optional summary for your report
# =========================

summary = pd.DataFrame({
    "Scenario": ["Low", "Mid", "High"],
    "Total Water (L/year)": [
        final_df["Water Consumption Low (L/year)"].sum(),
        final_df["Water Consumption Mid (L/year)"].sum(),
        final_df["Water Consumption High (L/year)"].sum()
    ],
    "Total Water (Gal/year)": [
        final_df["Water Consumption Low (Gal/year)"].sum(),
        final_df["Water Consumption Mid (Gal/year)"].sum(),
        final_df["Water Consumption High (Gal/year)"].sum()
    ]
})

display(summary)

,Scenario,Total Water (L/year),Total Water (Gal/year)
0,Low,2.162529e+10,5.712796e+09
1,Mid,2.883372e+10,7.617061e+09
2,High,3.604215e+10,9.521326e+09
